In [ ]:
import numpy as np
import pandas as pd

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv("https://raw.githubusercontent.com/aliraza5490/HousePricesEDA/refs/heads/main/data/train.csv")

In [ ]:
display(df.head())

In [ ]:
df.shape

In [ ]:
missing_percentages = (df.isnull().sum() / len(df)) * 100
missing_percentages = missing_percentages.drop(missing_percentages[missing_percentages == 0].index).sort_values(ascending=False)
display(missing_percentages)

In [ ]:
df = df.drop(["Id", "PoolQC", "MiscFeature", "Alley", "Fence", "MasVnrType", "FireplaceQu"], axis=1)

In [ ]:
for col in df.columns:
  if df[col].isnull().sum() > 0:
    print(col, df[col].isnull().sum())

In [ ]:
total_rows_with_missing_values = df.isnull().any(axis=1).sum()
print(f"Total rows with missing values: {total_rows_with_missing_values}")

In [ ]:
plt.figure(figsize=(10, 6))
sns.boxplot(x=df['LotFrontage'])
plt.title('Boxplot of LotFrontage')
plt.show()

In [ ]:
sorted_lot_frontage = df['LotFrontage'].sort_values()
df["LotFrontage"] = df["LotFrontage"].fillna(sorted_lot_frontage.median())

In [ ]:
df = df.dropna()
print("DataFrame shape after dropping missing values:", df.shape)

In [ ]:
for col in df.columns:
  if df[col].isnull().sum() > 0:
    print(col, df[col].isnull().sum())
else:
  print("No missing values remaining in the DataFrame.")

In [ ]:
df.shape

In [ ]:
df[df.duplicated()]

In [ ]:
df.head()

In [ ]:
df.tail()

In [ ]:
# check spread of lot frontange
df.describe()

In [ ]:
# Filter for numerical columns only
numeric_df = df.select_dtypes(include=[np.number])

# Calculate correlations
corr_matrix = numeric_df.corr()

# Plot heatmap
plt.figure(figsize=(15, 10))
sns.heatmap(corr_matrix, annot=False, linewidths=0.5)
plt.title('Correlation Heatmap of Numerical Features')
plt.show()

In [ ]:
for col in df.columns:
  print(col, df[col].dtypes)

In [ ]:
for column in df.columns:
    if df[column].dtype == 'object':
        df[column] = df[column].astype('category')

print('Categorical columns converted to category dtype.')

In [ ]:
df_encoded = pd.get_dummies(df, drop_first=True)

print('DataFrame shape after one-hot encoding:', df_encoded.shape)
display(df_encoded.head())

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

# Prepare features and target
X = df_encoded.drop('SalePrice', axis=1)
y = df_encoded['SalePrice']

# Split into training and validation sets to check 'accuracy'
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Initialize and train the model
rf_model = RandomForestRegressor(n_estimators=200, random_state=42)
rf_model.fit(X_train, y_train)

# Evaluate
y_pred = rf_model.predict(X_val)
r2 = r2_score(y_val, y_pred)
mae = mean_absolute_error(y_val, y_pred)
rmse = np.sqrt(mean_squared_error(y_val, y_pred))

print(f'R-squared Score (Accuracy): {r2:.4f}')
print(f'Mean Absolute Error: ${mae:.2f}')
print(f'Root Mean Squared Error: ${rmse:.2f}')


In [ ]:
# Load test data
test_df = pd.read_csv('https://raw.githubusercontent.com/aliraza5490/HousePricesEDA/refs/heads/main/data/test.csv')

# Preprocess test data (matching training steps)
test_df_clean = test_df.drop(["Id", "PoolQC", "MiscFeature", "Alley", "Fence", "MasVnrType", "FireplaceQu"], axis=1, errors='ignore')
test_df_clean["LotFrontage"] = test_df_clean["LotFrontage"].fillna(test_df_clean["LotFrontage"].mean())

# One-hot encode and align columns
test_encoded = pd.get_dummies(test_df_clean)

# Align test columns with training columns
test_encoded = test_encoded.reindex(columns=X.columns, fill_value=0)

# Predict
test_predictions = rf_model.predict(test_encoded)

# Show results
results = pd.DataFrame({'Id': test_df['Id'], 'PredictedPrice': test_predictions})
display(results.head())

In [27]:
import xgboost as xgb
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import numpy as np

# Initialize the XGBoost Regressor
# Using some standard hyperparameters
xgb_model = xgb.XGBRegressor(n_estimators=1000, learning_rate=0.05, max_depth=5, random_state=42)

# Train the model
xgb_model.fit(X_train, y_train,
              eval_set=[(X_val, y_val)],
              verbose=False)

# Predict and evaluate
y_pred_xgb = xgb_model.predict(X_val)
r2_xgb = r2_score(y_val, y_pred_xgb)
mae_xgb = mean_absolute_error(y_val, y_pred_xgb)
rmse_xgb = np.sqrt(mean_squared_error(y_val, y_pred_xgb))

print('XGBoost Regressor Performance:')
print(f'R-squared Score: {r2_xgb:.4f}')
print(f'Mean Absolute Error: ${mae_xgb:.2f}')
print(f'Root Mean Squared Error: ${rmse_xgb:.2f}')

XGBoost Regressor Performance:
R-squared Score: 0.8643
Mean Absolute Error: $15403.09
Root Mean Squared Error: $24098.20


In [29]:
# Load test data
test_df = pd.read_csv('https://raw.githubusercontent.com/aliraza5490/HousePricesEDA/refs/heads/main/data/test.csv')

# Preprocess test data (matching training steps)
test_df_clean = test_df.drop([ "PoolQC", "MiscFeature", "Alley", "Fence", "MasVnrType", "FireplaceQu"], axis=1, errors='ignore')
test_df_clean["LotFrontage"] = test_df_clean["LotFrontage"].fillna(test_df_clean["LotFrontage"].mean())

# One-hot encode and align columns
test_encoded = pd.get_dummies(test_df_clean)

# Align test columns with training columns
test_encoded = test_encoded.reindex(columns=X.columns, fill_value=0)

test_predictions_xgb = xgb_model.predict(test_encoded)

submission_xgb = pd.DataFrame({'Id': test_df['Id'], 'SalePrice': test_predictions_xgb})
submission_xgb.to_csv('/content/submission_xgb.csv', index=False)

print('Submission file created successfully at /content/submission_xgb.csv')
display(submission_xgb.head())

Submission file created successfully at /content/submission_xgb.csv


,Id,SalePrice
0,1461,126795.679688
1,1462,162135.406250
2,1463,187716.953125
3,1464,193898.093750
4,1465,202875.421875
